# LinearPartition
<!-- SPDX-License-Identifier: GPL-3.0-only -->

Adapted from `sinc-lab/lncRNA-folding`.
Modified by Jingwen Liu, 2026, for full-length viral RNA benchmarking.

In [ ]:
import os
import pandas as pd
import time
from pathlib import Path
import datetime

In [ ]:
method_name = "LinearPartition"
base = Path.cwd()

linearpartition_dir = base.parent / 'tools' / 'LinearPartition'
linearpartition_script = linearpartition_dir / 'linearpartition'

if not linearpartition_dir.exists():
    print("install LinearPartition...")
    os.chdir('../tools')
    !git clone https://github.com/LinearFold/LinearPartition.git
    os.chdir('LinearPartition')
    !make
    os.chdir('../../methods')
    print('LinearPartition built successfully')
else:
    print('LinearPartition already exists')

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
def run_folding(fasta_name, linearpartition_path, virus_id):
    temp_dot_file = f'LinearPartition_tmp_{virus_id.replace("#", "_").replace("/", "_")}.dot'
    cmd = f'cat {fasta_name} | {linearpartition_path} -V -M > {temp_dot_file} 2>/dev/null'
    result = os.system(cmd)
    
    if result != 0:
        return None, temp_dot_file

    # Reading sequence information
    with open(f'{fasta_name}', 'r') as fp:
        lines = fp.readlines()
        name, sequence = lines[0], lines[1]
    
    # Reading prediction
    with open(temp_dot_file, 'r') as fp:
        content = fp.read().strip()
        lines = content.split('\n')
        prediction = lines[-1]

    # Building output
    out_file_name = f"LinearPartition_clean_tmp_{virus_id.replace('#', '_').replace('/', '_')}.dot"
    with open(out_file_name, "w") as out_file:
        out_file.write(f'{name}{sequence}{prediction}\n')

    return out_file_name, temp_dot_file

In [ ]:
out_dir = Path.cwd().parent / 'prediction'
os.makedirs(out_dir, exist_ok=True)
out_fasta_path = out_dir / (method_name + ".fasta")

if out_fasta_path.exists():
    os.remove(out_fasta_path)

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}\t{'status'}")
successful_runs = 0

for i, vid in enumerate(virus_ids): 
    start_time = time.time()
    seq = viruses.loc[vid]["sequence"]
    print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)

    # Write a one-sequence fasta
    temp_fasta = f"LinearPartition_tmp_{vid.replace('#', '_').replace('/', '_')}.fasta"
    with open(temp_fasta, "w") as ofile: 
        ofile.write(f">{vid}\n{seq}\n")
    
    dot_file_name, temp_dot_file = run_folding(temp_fasta, str(linearpartition_script), vid)
    elapsed_time = time.time() - start_time
    
    if dot_file_name and os.path.exists(dot_file_name):
        # Concatenate outputs
        with open(dot_file_name, 'r') as infile:
            with open(out_fasta_path, 'a') as outfile:
                outfile.write(infile.read())
        successful_runs += 1
        status = "SUCCESS"
    else:
        status = "FAILED"

    print(f"{elapsed_time: .1f} s\t{status}")
    
    for tmp_file in [temp_fasta, dot_file_name, temp_dot_file]:
        if tmp_file and os.path.exists(tmp_file):
            os.remove(tmp_file)

print(f"\nCompleted: {successful_runs}/{len(virus_ids)} sequences processed successfully")
print(f"Output file: {out_fasta_path}")